In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import os

In [4]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [28]:
root = "c:\\Architectural-Biases-in-Time-Series-Anomaly-Detection"
def create_dict(name):
    path = os.path.join(root, "saved_model_scores", name)
    with np.load(path) as data:
       results = {name: metric for name, metric in data.items()}
    return results
lstm_f_results = create_dict("lstm_f_results.npz")
lstm_ae_results = create_dict("lstm_ae_results.npz")
transformer_f_results = create_dict("transformer_f_results.npz")

In [95]:
transformer_f_results

{'v_scores': array([nan, nan, nan, ..., nan, nan, nan], shape=(2499740,), dtype=float32),
 'v_labels': array([0, 0, 0, ..., 0, 0, 0], shape=(2499740,)),
 'v_categories': array([0., 0., 0., ..., 0., 0., 0.], shape=(2499740,)),
 't_scores': array([nan, nan, nan, ..., nan, nan, nan], shape=(1499740,), dtype=float32),
 't_labels': array([0, 0, 0, ..., 0, 0, 0], shape=(1499740,)),
 't_categories': array([0., 0., 0., ..., 0., 0., 0.], shape=(1499740,))}

In [92]:
ae_scores = lstm_ae_results["v_scores"]
ae_labels = lstm_ae_results["v_labels"]
ae_cats = lstm_ae_results["v_categories"]

window_grid = np.linspace(300, 800, 5)
for window in window_grid:
    window = int(window)
    if window >= 1:
        current_score = np.convolve(ae_scores, np.ones(window) / window, mode='same')

    threshold = np.percentile(current_score, 100 - 4.4)
    TP = np.sum((current_score > threshold) & (ae_labels == 1))
    FP = np.sum((current_score > threshold) & (ae_labels == 0))
    recall = TP/np.sum(ae_labels == 1)
    precision = TP/(TP + FP)
    F1 = 2 * (precision * recall)/(precision + recall)
    print(f"| window: {window} | recall {recall:.2f}, precision {precision:.2f}, F1 {F1:.4f}")

| window: 300 | recall 0.44, precision 0.63, F1 0.5157
| window: 425 | recall 0.45, precision 0.64, F1 0.5290
| window: 550 | recall 0.45, precision 0.65, F1 0.5353
| window: 675 | recall 0.46, precision 0.65, F1 0.5372
| window: 800 | recall 0.46, precision 0.65, F1 0.5376


In [70]:
f_scores = lstm_f_results["v_scores"]
f_labels = lstm_f_results["v_labels"]
f_cats = lstm_f_results["v_categories"]

window_grid = np.linspace(300, 800, 5)
for window in window_grid:
    window = int(window)
    if window >= 1:
        current_score = np.convolve(f_scores, np.ones(window) / window, mode='same')

    threshold = np.percentile(current_score, 100 - 4)
    TP = np.sum((current_score > threshold) & (f_labels == 1))
    FP = np.sum((current_score > threshold) & (f_labels == 0))
    recall = TP/np.sum(labels == 1)
    precision = TP/(TP + FP)
    F1 = 2 * (precision * recall)/(precision + recall)
    print(f"| window: {window} | recall {recall:.2f}, precision {precision:.2f}, F1 {F1:.4f}")

| window: 300 | recall 0.52, precision 0.66, F1 0.5813
| window: 425 | recall 0.54, precision 0.68, F1 0.6044
| window: 550 | recall 0.54, precision 0.68, F1 0.6030
| window: 675 | recall 0.52, precision 0.66, F1 0.5824
| window: 800 | recall 0.49, precision 0.62, F1 0.5492


In [93]:
def generate_df(scores, labels, cats, window_grid):
    series = []

    for window in window_grid:
        window = int(window)
        current_score = np.convolve(scores, np.ones(window) / window, mode='same')
        threshold = np.percentile(current_score, 100 - 4.4)
        cat_dict = {}
        normal_mask = cats == 0
        FPR = np.sum(current_score[normal_mask] > threshold) / np.sum(normal_mask)
        cat_dict["FPR (0.0)"] = round(float(FPR), 2)

        for cat in np.unique(cats):
            if cat == 0.0:
                continue
            cat_mask = cats == cat
            recall = np.sum(current_score[cat_mask] > threshold) / np.sum(cat_mask)
            cat_dict[f"recall ({cat})"] = round(float(recall), 2)

        TP = np.sum((current_score > threshold) & (labels == 1))
        FP = np.sum((current_score > threshold) & (labels == 0))
        recall = TP / np.sum(labels == 1)
        precision = TP / (TP + FP)
        F1 = 2 * (precision * recall) / (precision + recall)
        cat_dict["F1"] = np.round(F1, 3)

        series.append(pd.Series(cat_dict, name=f"t={threshold:.2f}|w={window}"))

    return pd.concat(series, axis=1)

generate_df(f_scores, f_labels, f_cats, window_grid)

,t=27.67|w=300,t=27.63|w=425,t=29.66|w=550,t=32.40|w=675,t=39.97|w=800
FPR (0.0),0.020,0.020,0.020,0.020,0.020
recall (1.0),0.910,0.930,0.960,0.980,0.990
recall (2.0),0.330,0.340,0.350,0.330,0.350
recall (3.0),0.950,0.980,1.000,0.910,0.910
recall (4.0),0.590,0.640,0.680,0.680,0.700
recall (5.0),0.010,0.020,0.040,0.050,0.060
recall (6.0),0.310,0.360,0.380,0.310,0.270
recall (7.0),0.510,0.550,0.550,0.530,0.480
recall (8.0),0.850,0.850,0.860,0.870,0.850
recall (9.0),0.940,0.930,0.910,0.860,0.790


In [94]:
generate_df(ae_scores, ae_labels, ae_cats, window_grid)

,t=5641.50|w=300,t=5437.10|w=425,t=5338.10|w=550,t=5281.23|w=675,t=5245.51|w=800
FPR (0.0),0.020,0.020,0.020,0.020,0.020
recall (1.0),0.930,0.940,0.970,0.970,0.980
recall (2.0),0.220,0.230,0.240,0.220,0.160
recall (3.0),0.640,0.630,0.660,0.680,0.690
recall (4.0),0.530,0.560,0.570,0.580,0.590
recall (5.0),0.110,0.120,0.120,0.090,0.120
recall (6.0),0.290,0.260,0.230,0.230,0.230
recall (7.0),0.470,0.490,0.490,0.490,0.490
recall (8.0),0.690,0.690,0.700,0.700,0.690
recall (9.0),0.640,0.650,0.660,0.650,0.650
